In [34]:
import pandas as pd
from itertools import combinations

def simulate_match(player1, player2, results_df):
    """
    Given two players, look up their predetermined match result.
    The results_df has one row per unordered pair, with:
      - player_id_1: the smaller id
      - player_id_2: the larger id
      - score: 1 means player_id_1 wins, 0 means player_id_2 wins.

    Returns a dictionary with the result: win=1, loss=0.
    """
    p_low, p_high = sorted([player1, player2])
    match_row = results_df[
        (results_df['player_id_1'] == p_low) & (results_df['player_id_2'] == p_high)
    ]
    if match_row.empty:
        raise ValueError(f"No match result found for players {p_low} and {p_high}")

    outcome = match_row.iloc[0]['score']
    if outcome == 1:
        # p_low wins.
        if player1 == p_low:
            return {player1: 1, player2: 0}
        else:
            return {player1: 0, player2: 1}
    elif outcome == 0:
        # p_high wins.
        if player1 == p_low:
            return {player1: 0, player2: 1}
        else:
            return {player1: 1, player2: 0}
    else:
        raise ValueError("Invalid outcome value. Only 1 or 0 are allowed.")

def pair_players(sorted_players, opponents_record):
    """
    Attempt to pair players (given as a list) so that no pairing repeats an earlier game.
    Uses recursive backtracking.

    Returns a list of pairs [(p1, p2), ...] or None if no valid pairing is found.
    """
    if not sorted_players:
        return []

    first = sorted_players[0]
    for i in range(1, len(sorted_players)):
        candidate = sorted_players[i]
        if candidate not in opponents_record[first]:
            remaining = sorted_players[1:i] + sorted_players[i+1:]
            rest_pairing = pair_players(remaining, opponents_record)
            if rest_pairing is not None:
                return [(first, candidate)] + rest_pairing
    return None  # no valid pairing found

def greedy_pairing(players):
    """
    Fallback pairing: simply pairs players sequentially from the list.
    """
    pairs = []
    for i in range(0, len(players), 2):
        pairs.append((players[i], players[i+1]))
    return pairs

def simulate_swiss_until_threshold(results_df, n):
    """
    Simulate a tournament where each player continues playing until they have either
    n wins or n losses. In each round, the active players (those with wins < n and losses < n)
    are grouped by their win count (only players with the same win–loss record may face each other).

    If a group has an odd number of active players, one player gets a bye (which counts as a win).

    The predetermined outcomes for matchups are looked up from results_df.

    Returns a tuple: (final_standings DataFrame, total number of matches played).
    """
    # Get the set of all players.
    players = set(results_df['player_id_1']).union(set(results_df['player_id_2']))

    # Initialize records.
    wins = {p: 0 for p in players}
    losses = {p: 0 for p in players}
    opponents_record = {p: set() for p in players}
    byes = {p: 0 for p in players}  # Count of byes (a bye counts as a win)

    match_counter = 0  # Counter for total number of matches played

    round_num = 0
    while True:
        # Active players: those who have not yet reached n wins or n losses.
        active_players = [p for p in players if wins[p] < n and losses[p] < n]

        # If fewer than 2 active players remain, we cannot form a match.
        if len(active_players) < 2:
            if active_players:
                lone_player = active_players[0]
                print(f"Round {round_num+1}: Only one active player ({lone_player}) remains. Giving bye.")
                wins[lone_player] += 1
                byes[lone_player] += 1
            break

        round_num += 1
        print(f"\n=== Round {round_num} ===")

        # Group active players by their current win count.
        groups = {}
        for p in active_players:
            groups.setdefault(wins[p], []).append(p)

        round_pairings = []
        # Process each win group separately.
        for win_record in sorted(groups.keys(), reverse=True):
            group_players = groups[win_record]
            group_players = sorted(group_players)  # sort for determinism

            # If the group has an odd number of players, assign a bye.
            if len(group_players) % 2 == 1:
                bye_player = group_players[0]
                print(f"Round {round_num}: Player {bye_player} (win record: {wins[bye_player]}) gets a bye.")
                wins[bye_player] += 1
                byes[bye_player] += 1
                group_players.remove(bye_player)

            # Pair the remaining players in this win group.
            if group_players:
                pairing = pair_players(group_players, opponents_record)
                if pairing is None:
                    print(f"Warning: No pairing without rematches found in win group {win_record}; using greedy pairing.")
                    pairing = greedy_pairing(group_players)
                round_pairings.extend(pairing)

        # Simulate the matches for this round.
        for p1, p2 in round_pairings:
            result = simulate_match(p1, p2, results_df)
            match_counter += 1  # Increment the match counter for each match played
            if result[p1] == 1:
                wins[p1] += 1
                losses[p2] += 1
                winner = p1
            else:
                wins[p2] += 1
                losses[p1] += 1
                winner = p2
            # Record that these two players have met.
            opponents_record[p1].add(p2)
            opponents_record[p2].add(p1)
            print(f"Round {round_num}: Match: {p1} vs {p2} -> Winner: {winner}")

        # Print standings after the round.
        print(f"Standings after round {round_num}:")
        for p in sorted(players, key=lambda x: (-wins[x], wins[x] - losses[x], x)):
            print(f"  Player {p}: {wins[p]} wins, {losses[p]} losses, byes: {byes[p]}")

    # Build final standings DataFrame.
    final_standings = pd.DataFrame(
        [{"player_id": p, "wins": wins[p], "losses": losses[p], "byes": byes[p]} for p in players]
    )
    final_standings = final_standings.sort_values(by=["wins", "player_id"], ascending=[False, True]).reset_index(drop=True)
    print(f"\nTotal matches played: {match_counter}")
    return final_standings, match_counter

# ---------------------------
# Example usage:
# ---------------------------
if __name__ == "__main__":
    # For example, suppose we have 6 players.
    players = list(range(1, 1001, 1))
    rows = []
    # Create all possible pairings (with player_id_1 < player_id_2).
    # For demonstration, let the predetermined outcome be:
    #   If player_id_1 is even then player_id_1 wins (score=1); otherwise, player_id_2 wins (score=0).
    for p1, p2 in combinations(players, 2):
        if p1 % 2 == 0:
            score = 1  # p1 wins.
        else:
            score = 0  # p2 wins.
        rows.append({"player_id_1": p1, "player_id_2": p2, "score": score})
    results_df = pd.DataFrame(rows)

    n = 10  # Each player plays until they have 4 wins or 4 losses.
    final_standings, total_matches = simulate_swiss_until_threshold(results_df, n)

    print("\nFinal Standings:")
    print(final_standings)
    print(f"\nTotal matches played: {total_matches}")


Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
  Player 491: 4 wins, 10 losses, byes: 0
  Player 507: 4 wins, 10 losses, byes: 0
  Player 547: 4 wins, 10 losses, byes: 0
  Player 549: 4 wins, 10 losses, byes: 0
  Player 591: 4 wins, 10 losses, byes: 0
  Player 599: 4 wins, 10 losses, byes: 0
  Player 635: 4 wins, 10 losses, byes: 0
  Player 641: 4 wins, 10 losses, byes: 0
  Player 671: 4 wins, 10 losses, byes: 0
  Player 677: 4 wins, 10 losses, byes: 0
  Player 723: 4 wins, 10 losses, byes: 0
  Player 743: 4 wins, 10 losses, byes: 0
  Player 757: 4 wins, 10 losses, byes: 0
  Player 797: 4 wins, 10 losses, byes: 0
  Player 799: 4 wins, 10 losses, byes: 0
  Player 821: 4 wins, 10 losses, byes: 0
  Player 869: 4 wins, 10 losses, byes: 0
  Player 871: 4 wins, 10 losses, byes: 0
  Player 883: 4 wins, 10 losses, byes: 0
  Player 885: 4 wins, 10 losses, byes: 0
  Player 945: 4 wins, 10 losses, byes: 0
  Player 947: 4 wins, 10 losses, byes: 0
  Player 5: 3 wins, 10 losses, by

In [ ]:
final_standings

,player_id,wins,losses,byes
0,2,10,0,3
1,4,10,1,5
2,6,10,2,3
3,7,10,6,7
4,8,10,2,0
...,...,...,...,...
795,691,2,10,0
796,33,1,10,1
797,35,1,10,0
798,289,1,10,1


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=final_standings)